# IIC-3641 GML UC

## Actividad en clase

Vamos a usar el Graph Transformer para trabajar en esta actividad.

Trabajaremos con **`ppbr_az`** del ADMET Group del TDC: el porcentaje de fármaco que queda **unido a proteínas plasmáticas** (0 a 100), medido por AstraZeneca en 1.797 moléculas. Ojo con dos cosas: una misma molécula aparece **varias veces**, medida en distintas especies (humano, rata, perro, ratón...), y la distribución está muy cargada hacia arriba: la mitad de las mediciones está sobre 94%.

- Corra el notebook del Graph Transformer y verifique que puede reproducir el ejemplo de la clase (lipofilicidad).
- Cargue `ppbr_az` con el **test oficial** del benchmark y saque la validación con el **split por scaffold** que vimos en clases. Antes de entrenar, calcule el MAE de test de un modelo que **no mira la molécula** y predice siempre la mediana de train. Ése es el piso.
- Reentrene el graph transformer. Pruebe variantes con distinto número de cabezales y capas (incluya la de la clase) para ver el efecto en el MAE. Haga 3 variantes, al menos, corra 100 epochs y use **al menos 3 semillas por variante**. Reporte media ± desviación estándar del MAE de test en la época de mejor validación. ¿Las diferencias entre variantes son más grandes que la variación entre semillas?
- Vea la última lámina de la clase y dígame si le ganó al paper. ¿Y a la mediana?
- El año pasado esta actividad se hizo con un **split aleatorio** 70/10/20, y los modelos le ganaban al paper con holgura. Repita su mejor variante con un split aleatorio sobre todas las filas (train_val + test) y explique la diferencia. Pista: cuente cuántas moléculas del test también aparecen en train.
- Cuanto termine, me avisa para entregarle una **L (logrado)**.
- Recuerde que las L otorgan un bono en la nota final de la asignatura.

***Tiene hasta el final de la clase.***

In [1]:
import os, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import torch
from torch import nn
from rdkit import RDLogger
from torch_geometric.loader import DataLoader
from gt_pyg import GraphTransformerNet, get_tensor_data

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore', category=UserWarning)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ADMET_ZIP = 'data/admet_group.zip'
if not os.path.exists(ADMET_ZIP):
    os.makedirs('data', exist_ok=True)
    req = urllib.request.Request('https://dataverse.harvard.edu/api/access/datafile/4426004',
                                 headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as r, open(ADMET_ZIP, 'wb') as f:
        f.write(r.read())

with zipfile.ZipFile(ADMET_ZIP) as z:
    train_val = pd.read_csv(z.open('admet_group/ppbr_az/train_val.csv'))
    test = pd.read_csv(z.open('admet_group/ppbr_az/test.csv'))

print(f'train_val: {len(train_val)} filas, {train_val.Drug.nunique()} moléculas distintas')
print(f'test:      {len(test)} filas, {test.Drug.nunique()} moléculas distintas')
train_val.head()

train_val: 2231 filas, 1454 moléculas distintas
test:      559 filas, 343 moléculas distintas


,Drug_ID,Drug,Y
0,CHEMBL2316382,COc1ccc(NC(=O)c2ccc(C[S+](C)[O-])c3ccccc23)c(C...,94.56
1,CHEMBL2316382,COc1ccc(NC(=O)c2ccc(C[S+](C)[O-])c3ccccc23)c(C...,95.63
2,CHEMBL2316383,C[S+]([O-])Cc1ccc(C(=O)Nc2cccnc2C(=O)NCC2CCOCC...,84.60
3,CHEMBL472,COc1ccc(Cl)cc1C(=O)NCCc1ccc(S(=O)(=O)NC(=O)NC2...,99.73
4,CHEMBL472,COc1ccc(Cl)cc1C(=O)NCCc1ccc(S(=O)(=O)NC(=O)NC2...,99.88


### Continúe...